In [1]:
import os
import google.generativeai as genai
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
genai.configure(api_key=os.getenv("GEMINI_API_KEY"))

def get_completion_from_messages(messages, 
                                 model="gemini-3.1-flash-lite-preview", 
                                 temperature=0, 
                                 max_tokens=500):
    # Extract the system message if it exists
    system_content = ""
    chat_history = []
    
    for msg in messages:
        if msg['role'] == 'system':
            system_content = msg['content']
        else:
            # Map OpenAI roles to Gemini roles
            role = "model" if msg["role"] == "assistant" else "user"
            chat_history.append({"role": role, "parts": [msg["content"]]})

    # Initialize model with the system instruction
    gemini_model = genai.GenerativeModel(
        model_name=model,
        system_instruction=system_content,
        generation_config={
            "temperature": temperature,
            "max_output_tokens": max_tokens,
            "response_mime_type": "application/json" # Ensures valid JSON list output
        }
    )

    # Separate the last message as the active prompt
    if chat_history:
        last_msg = chat_history.pop()["parts"][0]
        chat = gemini_model.start_chat(history=chat_history)
        response = chat.send_message(last_msg)
    else:
        # Fallback if only system message was provided
        response = gemini_model.generate_content("Proceed.")

    return response.text

# --- Your Logic ---
delimiter = "####"
system_message = f"""
You will be provided with customer service queries. The customer service query will be delimited with {delimiter} characters.
Output a python list of objects, where each object has the following format:
    'category': <one of Computers and Laptops, Smartphones and Accessories, Televisions and Home Theater Systems, Gaming Consoles and Accessories, Audio Equipment, Cameras and Camcorders>,
OR
    'products': <a list of products that must be found in the allowed products below>

[... rest of your allowed products list ...]
Only output the list of objects, with nothing else.
"""

user_message_1 = f"tell me about the smartx pro phone and the fotosnap camera, the dslr one. Also tell me about your tvs"

messages = [  
    {'role': 'system', 'content': system_message},  
    {'role': 'user', 'content': f"{delimiter}{user_message_1}{delimiter}"},  
] 

category_and_product_response_1 = get_completion_from_messages(messages)
print(category_and_product_response_1)

C:\Anaconda3\envs\lang_chain\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\User\AppData\Local\Temp\ipykernel_19900\3512798362.py:2: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


[
  {
    "products": [
      "SmartX Pro phone",
      "FotoSnap camera"
    ]
  },
  {
    "category": "Televisions and Home Theater Systems"
  }
]
